# M2: lấy mẫu và EDA song ngữ (CPU)

Notebook này điều phối module Python trong repo. Chạy thử với 1.000 hàng trước; sau đó đổi `SAMPLE_SIZE` thành 10.000 nếu quy trình ổn. Không cần GPU hoặc Hugging Face token. Snapshot và báo cáo lưu trong Google Drive, không đưa dataset vào Git. Lấy mẫu theo cụm 100 hàng, không phải IID từng hàng.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/coder-fishing/KidStory.git"
BRANCH = "main"
DATASET_REVISION = "eb37a7aa8cc44ad9841175163a460fed5c2c1b24"
SAMPLE_SIZE = 1000  # Pilot. Đổi thành 10000 sau khi pilot ổn.
SEED = 42
REPO_DIR = Path("/content/kids-story")
DRIVE_ROOT = Path("/content/drive/MyDrive/kids-story-runs/m2")
RUN_STARTED_AT = datetime.now(timezone.utc)
RUN_ID = f"eda_{SAMPLE_SIZE}_seed{SEED}_{RUN_STARTED_AT:%Y%m%dT%H%M%SZ}"
RUN_DIR = DRIVE_ROOT / RUN_ID
SAMPLE_DIR = RUN_DIR / "source"
EDA_DIR = RUN_DIR / "eda"

if SAMPLE_SIZE not in {1000, 10000}:
    raise ValueError("SAMPLE_SIZE phải là 1000 hoặc 10000")

## 1. Lấy mã nguồn đã push lên GitHub
Trước khi chạy notebook, commit và push thay đổi muốn dùng lên `main`. Notebook clone repo public, không cần token. Hãy mở runtime mới nếu thư mục repo đã tồn tại.

In [ ]:
import subprocess

if REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} đã tồn tại; hãy dùng runtime Colab mới để tránh chạy nhầm phiên bản.")
clone = subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPOSITORY_URL, str(REPO_DIR)], capture_output=True, text=True)
if clone.returncode:
    raise RuntimeError(clone.stderr or "Git clone thất bại. Kiểm tra mạng và nhánh.")
source_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip()
print("Repo commit:", source_commit)

## 2. Kết nối Drive và chuẩn bị module CPU
Mỗi lần chạy có `RUN_ID` UTC riêng. Log ghi cả lỗi nếu lấy mẫu hoặc EDA thất bại; artifact vẫn nằm cùng thư mục trên Drive.

In [ ]:
from google.colab import drive
import json
import platform
import sys

drive.mount("/content/drive")
sys.path.insert(0, str(REPO_DIR / "src"))
from kids_story.data.source import sample_source
from kids_story.data.eda import run_eda
RUN_DIR.mkdir(parents=True, exist_ok=False)
EVENT_LOG = RUN_DIR / "run.log"
def log_event(event, **details):
    entry = {"at_utc": datetime.now(timezone.utc).isoformat(), "event": event, **details}
    with EVENT_LOG.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(entry)
log_event("notebook_started", run_id=RUN_ID, sample_size=SAMPLE_SIZE, seed=SEED, source_commit=source_commit)
print("Python module sẵn sàng. CPU-only; không tải model.")

## 3. Lấy snapshot từ revision đã khóa
Mỗi request tối đa 100 hàng; notebook lấy các block 100 hàng rải theo seed. Mọi response được kiểm tra `x-revision`; nếu revision không khớp, cell dừng.

In [ ]:
try:
    sample_manifest = sample_source(
        SAMPLE_DIR, revision=DATASET_REVISION, limit=SAMPLE_SIZE,
        method="random_blocks", seed=SEED,
    )
except Exception as exc:
    log_event("source_sampling_failed", error_type=type(exc).__name__, error=str(exc))
    raise
log_event("source_sampling_complete", revision=sample_manifest["resolved_revision"],
          sampling=sample_manifest["sampling"]["method"], rows=len(sample_manifest["sampling"]["row_indices"]))
print({"revision": sample_manifest["resolved_revision"],
       "sampling": sample_manifest["sampling"]["method"],
       "rows": len(sample_manifest["sampling"]["row_indices"]),
       "sample_dir": str(SAMPLE_DIR)})

## 4. Chạy EDA và tạo bảng để review
Báo cáo thống kê en/vi, exact duplicate, nhóm nguồn, dấu hiệu cắt cụt/lặp và ký tự bất thường. CSV tên/dịch sát chỉ là ứng viên `pending`; notebook không sửa nội dung truyện.

In [ ]:
try:
    report = run_eda(
        SAMPLE_DIR / "sample.jsonl", EDA_DIR,
        source_manifest_path=SAMPLE_DIR / "source_manifest.json",
    )
except Exception as exc:
    log_event("eda_failed", error_type=type(exc).__name__, error=str(exc))
    raise
log_event("eda_complete", rows=report["input_rows"],
          name_candidates=report["name_candidate_rows"],
          translation_candidates=report["translation_calque_candidate_rows"])
print({"rows": report["input_rows"],
       "name_candidates": report["name_candidate_rows"],
       "translation_candidates": report["translation_calque_candidate_rows"],
       "report_dir": str(EDA_DIR)})

## 5. Vẽ biểu đồ và lưu PNG vào Drive
Biểu đồ mô tả phân bố độ dài, tín hiệu dữ liệu cơ bản và tần suất ứng viên. Đây là thống kê khảo sát, không phải đánh giá chất lượng.

In [ ]:
import csv
import json
from collections import Counter
import matplotlib.pyplot as plt

with (SAMPLE_DIR / "sample.jsonl").open(encoding="utf-8") as stream:
    sampled_rows = [json.loads(line) for line in stream if line.strip()]
lengths = {field: [len(row[field]) for row in sampled_rows if isinstance(row.get(field), str) and row[field].strip()] for field in ("en", "vi")}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, field in zip(axes, ("en", "vi")):
    axis.hist(lengths[field], bins=30, color="#356859", edgecolor="white")
    axis.set(title=f"{field}: character length", xlabel="Characters", ylabel="Rows")
fig.tight_layout()
length_chart = EDA_DIR / "text_length_distributions.png"
fig.savefig(length_chart, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

quality_labels = ["null", "empty", "duplicate excess", "missing terminal punctuation"]
quality_counts = {}
for field in ("en", "vi"):
    summary = report["field_summary"][field]
    quality_counts[field] = [summary["null_count"], summary["empty_or_whitespace_count"],
        summary["exact_duplicate_excess_after_nfc_whitespace_normalization"],
        summary["missing_terminal_punctuation_count"]]
fig, axis = plt.subplots(figsize=(10, 4))
x = list(range(len(quality_labels)))
width = 0.38
axis.bar([i - width/2 for i in x], quality_counts["en"], width, label="en", color="#356859")
axis.bar([i + width/2 for i in x], quality_counts["vi"], width, label="vi", color="#df9b42")
axis.set_xticks(x, quality_labels, rotation=15, ha="right")
axis.set_ylabel("Rows / duplicate excess")
axis.legend()
fig.tight_layout()
quality_chart = EDA_DIR / "data_quality_overview.png"
fig.savefig(quality_chart, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

with (EDA_DIR / "name_candidates.csv").open(encoding="utf-8-sig", newline="") as stream:
    name_counts = Counter(row["source_name_en"] for row in csv.DictReader(stream))
with (EDA_DIR / "translation_candidates.csv").open(encoding="utf-8-sig", newline="") as stream:
    rule_counts = Counter(row["rule_id"] for row in csv.DictReader(stream))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_names = name_counts.most_common(15)
if top_names:
    axes[0].barh([name for name, _ in top_names][::-1], [count for _, count in top_names][::-1], color="#356859")
axes[0].set(title="English name candidates", xlabel="Occurrences")
if rule_counts:
    axes[1].bar(list(rule_counts), list(rule_counts.values()), color="#df9b42")
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Translation pattern candidates", ylabel="Rows")
fig.tight_layout()
candidate_chart = EDA_DIR / "review_candidate_frequencies.png"
fig.savefig(candidate_chart, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
chart_paths = [length_chart, quality_chart, candidate_chart]
log_event("charts_saved", charts=[p.name for p in chart_paths])

## 6. Xem các bảng ứng viên
Tải các CSV trong `EDA_DIR` về để review hoặc mở bằng Google Sheets. Mỗi dòng cần quyết định theo ngữ cảnh; không coi gợi ý là bản dịch đã duyệt.

In [ ]:
from IPython.display import display
import pandas as pd

for filename in ("name_candidates.csv", "translation_candidates.csv"):
    path = EDA_DIR / filename
    table = pd.read_csv(path, encoding="utf-8-sig")
    print(filename, "rows=", len(table))
    display(table.head(20))
print("Artifact output:", EDA_DIR)

## 7. Ghi cấu hình, log và xác minh hash
Run-level manifest ghi commit mã, revision dữ liệu, seed, runtime CPU, hash snapshot/báo cáo/biểu đồ. `COMPLETE.json` kiểm tra toàn bộ file trong thư mục run.

In [ ]:
import hashlib
import os

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
finished_at = datetime.now(timezone.utc).isoformat()
run_config = {
    "schema_version": 1, "status": "complete", "run_id": RUN_ID,
    "started_at_utc": RUN_STARTED_AT.isoformat(), "finished_at_utc": finished_at,
    "repository_url": REPOSITORY_URL, "source_commit": source_commit,
    "dataset_revision": sample_manifest["resolved_revision"],
    "sampling": sample_manifest["sampling"],
    "sample_sha256": sha256_file(SAMPLE_DIR / "sample.jsonl"),
    "source_manifest_sha256": sha256_file(SAMPLE_DIR / "source_manifest.json"),
    "eda_report_sha256": sha256_file(EDA_DIR / "eda_report.json"),
    "charts": {path.name: sha256_file(path) for path in chart_paths},
    "runtime": {"python": platform.python_version(), "platform": platform.platform(),
                 "cpu_count": os.cpu_count(), "gpu_used": False},
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
log_event("run_complete", run_id=RUN_ID, run_config="run_config.json")
hashes = {str(path.relative_to(RUN_DIR)).replace("\\", "/"): sha256_file(path)
          for path in sorted(RUN_DIR.rglob("*")) if path.is_file() and path.name != "COMPLETE.json"}
(RUN_DIR / "COMPLETE.json").write_text(json.dumps(hashes, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Run hoàn tất. Xem log/config ở:", RUN_DIR)
print("Biểu đồ PNG:", [str(path) for path in chart_paths])